In [1]:
import numpy as np
import qiskit_metal as metal
import qiskit_metal as qm
from qiskit_metal import MetalGUI
from qiskit_metal import designs, draw
from qiskit_metal import Dict, open_docs
import pandas as pd

%metal_heading Welcome to Qiskit Metal!


In [ ]:
design = designs.DesignPlanar()
gui = MetalGUI(design)

02:23PM 02s WARNING [_qt_message_handler]: WARNING: Could not create pixmap from c:\Users�nji\Resonator-Frequency-Simulation\.venv\Lib\site-packages\qiskit_metal\_gui\styles\metal_dark\rc\transparent.png (No context available from Qt)
Python Traceback (most recent call last):
  File "c:\Users\Benji\Resonator-Frequency-Simulation\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Benji\Resonator-Frequency-Simulation\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\Benji\Resonator-Frequency-Simulation\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 807, in start
    self.io_loop.start()
  File "c:\Users\Benji\Resonator-Frequency-Simulation\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\Benji\AppData\Local\Programs\Python\Python312\Lib\asyncio\base_events.py", line 641, 

In [3]:
design.overwrite_enabled =  True

design.chips.main.size.size_x = '6.25mm'
design.chips.main.size.size_y = '6.25mm'
design.chips.main.size.size_z = '-270um'
design.chips.main.size.sample_holder_top = '50um'
design.chips.main.size.sample_holder_bottom = '800um'

In [4]:
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '6.25mm',
  'size_y': '6.25mm',
  'size_z': '-270um',
  'sample_holder_top': '50um',
  'sample_holder_bottom': '800um'}}

In [5]:
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven

In [6]:
port_1 = LaunchpadWirebondDriven(design,'port_1', options = dict(pos_x = '-2.496mm',
                                                                 pos_y = '-0mm',
                                                                 orientation = '0',
                                                                 pad_width = '350um',
                                                                 pad_height = '129um',
                                                                 pad_gap = '150um',
                                                                 trace_width = '10um',
                                                                 trace_gap = '6um',
                                                                 lead_length = '9um',
                                                                 taper_height = '350um'))

In [7]:
port_2 = LaunchpadWirebondDriven(design,'port_2',options = dict(pos_x = '2.496mm',
                                                                pos_y = '-0mm',
                                                                orientation = '180',
                                                                pad_width = '350um',
                                                                pad_height = '129um',
                                                                taper_height = '350um',
                                                                lead_length = '9um',
                                                                trace_width = '10um',
                                                                trace_gap = '6um',
                                                                pad_gap = '150um'))

In [8]:
gui.rebuild()

In [9]:
gui.autoscale()

In [10]:
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

In [11]:
print(design.components['port_1'].pins.keys())

dict_keys(['tie', 'in'])


In [12]:
bus_1 = RouteStraight(design,
                      'bus_1',
                      options = dict(hfss_wire_bonds = True,
                                     pin_inputs = dict(start_pin=dict(component='port_1',pin='tie'),
                                                        end_pin=dict(component='port_2',pin='tie')),
                                    trace_width = '10um',
                                    trace_gap = '6um',))

gui.rebuild()
gui.autoscale()

In [13]:
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

In [14]:
coupl_1 = CoupledLineTee(design,'coupl_1', options = dict(pos_x='-.55mm',
                                                          pos_y = '0',
                                                          orientation = '180',
                                                          coupling_length = '650um',
                                                          down_length = '100um',
                                                          fillet ='50um',
                                                          coupling_space ='4um',
                                                          open_termination = True))

gui.rebuild()
gui.autoscale()

In [15]:
from qiskit_metal.analyses.em.cpw_calculations import guided_wavelength

In [16]:
def find_resonator_length (frequency,line_width,line_gap,N):
    [lambdaG, etfSqrt, q] = guided_wavelength(frequency,line_width,
                                              line_gap, substrate_thickness=350*(10**-6),
                                              film_thickness=200*(10**-9))
    return str(lambdaG/N*10**3)+' mm'

In [17]:
#Half Wavelength Resonator
find_resonator_length(5*(10**9),6*(10**-6),10*(10**-6),2)

'12.14305707916776 mm'

In [18]:
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

In [19]:
open_end = OpenToGround(design, 'open_end',options=dict(pos_x='-0.55mm',
                                                        pos_y='2.25mm',
                                                        orientation='90',
                                                        termination_gap='6um',
                                                        gap='6um',
                                                        width = '10um'))

gui.rebuild()
gui.autoscale()

In [20]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

In [21]:
res_1 = RouteMeander(design, 'res_1', options = dict(
                                                    pin_inputs = dict(start_pin=dict(component = 'coupl_1',
                                                                                     pin = 'second_end'),
                                                                    end_pin=dict(component='open_end',
                                                                                 pin='open')),
                                                    lead =dict(start_straigt = '0um',
                                                               end_straight ='0um'),
                                                    meander = dict(spacing='150um',
                                                                   asymmetry='-300um'),
                                                    hfss_wire_bonds=True,
                                                    total_length='12mm',
                                                    trace_width='10um',
                                                    fillet='50um',
                                                    trace_gap='6um'))

gui.rebuild()
gui.autoscale()

In [22]:
# QDesign registers GDS renderer during init of QDesign.
a_gds = design.renderers.gds
# An alternate way to invoke gds commands without using a_gds:
# design.renderers.gds.export_to_gds()

# Show the options for GDS
a_gds.options

{'short_segments_to_not_fillet': 'True',
 'check_short_segments_by_scaling_fillet': '2.0',
 'gds_unit': 0.001,
 'ground_plane': 'True',
 'negative_mask': {'main': []},
 'fabricate': 'False',
 'corners': 'natural',
 'tolerance': '0.00001',
 'precision': '0.000000001',
 'width_LineString': '10um',
 'path_filename': '../resources/Fake_Junctions.GDS',
 'junction_pad_overlap': '5um',
 'max_points': '199',
 'cheese': {'datatype': '100',
  'shape': '0',
  'cheese_0_x': '25um',
  'cheese_0_y': '25um',
  'cheese_1_radius': '100um',
  'view_in_file': {'main': {1: True}},
  'delta_x': '100um',
  'delta_y': '100um',
  'edge_nocheese': '200um'},
 'no_cheese': {'datatype': '99',
  'buffer': '25um',
  'cap_style': '2',
  'join_style': '2',
  'view_in_file': {'main': {1: True}}},
 'bounding_box_scale_x': '1.2',
 'bounding_box_scale_y': '1.2'}

In [23]:
a_gds.export_to_gds("Single Resonator.gds")

02:22PM 58s WARNING [import_junction_gds_file]: Not able to find file:"../resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"c:\Users\Benji\resources".
02:22PM 59s WARNING [_cheese_buffer_maker]: The bounding box for no-cheese is outside of chip size.
Bounding box for chip is (-3.125, -3.125, 3.125, 3.125).
Bounding box with no_cheese buffer is (-3.15, -0.35000000000000003, 3.15, 2.2809999999999997).


1

In [24]:
design.chips['main']

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '6.25mm',
  'size_y': '6.25mm',
  'size_z': '-270um',
  'sample_holder_top': '50um',
  'sample_holder_bottom': '800um'}}

In [25]:
gui.figure.savefig('shot.png')

the expected resonant frequencu is $f_0 = 4.6385 \mathrm{GHz}$

klayout to qiskit metal

cursor

''hfss_renderer = design.renderers.hfss''
hfss_rendere.start()

Driven_mode = ScatteringImpedanceSim(design,'hfss')
hfss_drivenmode = Driven_mode.renderer

hfss_drivenmode.activate_ansys_design('Driven_Sim', 'drivenmodal')

hfss_drivenmode.render_design(selection = [], open_pins=[],port_list=[('port_1','in',50),('port_2','in',50)], box_plus_buffer = False''